# Task 3 — category summaries, hybrid approach

**Project 1 · Subha & Rayhan — Ironhack AI Engineering, AI FT SEPT 26**

The brief asks for an article per meta-category covering **the top 3 products, the common
complaints, and the worst product**.

Rayhan tried three times with a language model alone and was right to be unhappy with all
three:

| attempt | model | output |
|---|---|---|
| 1 | `flan-t5-small` | *"Great product. Able to connect a lot of different devices."* — one review copied |
| 2 | same | *"Overall opinion Do not copy individual reviews."* — **his own instruction handed back** |
| 3 | `Falconsai/text_summarization` | sentences stitched together, one about **Google Home** |

### Why this notebook does it differently

The failure was not the prompt. It was asking a language model for things it cannot know.

**"Top 3 products" and "worst product" are not opinions — they are `groupby` results.** A
model asked to infer them will invent them. So here:

| part of the article | where it comes from |
|---|---|
| review counts, ratings, sentiment split | `pandas` — exact |
| top 3 products, worst product | `pandas` — exact |
| common complaints | **log-odds** of words in negative vs positive reviews — measured |
| the evidence quote | the negative review most readers marked **helpful** |
| the connecting prose | a summarisation model, given only the quotes |

The model is used for the one thing it is actually good at: compressing a few sentences
into one. Everything a model could get wrong is computed instead.

**Runs on CPU.** Seconds without the model, a few minutes with it.

> **Note for Rayhan:** unlike `rayhan_task2_improved.ipynb`, this is a **new notebook**,
> not your file with edits. The approach is different enough that a diff would not have
> been readable. Instead every cell below is marked `### vs Rayhan's cell N` saying what
> you did there and why this does it differently. Your cell numbers refer to
> `rayhancreated.ipynb`.

In [6]:
%matplotlib inline

import json
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

warnings.simplefilter("default")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

CLUSTERED = next(c for c in [Path("../datasets/reviews_clustered.csv"),
                             Path("datasets/reviews_clustered.csv")] if c.exists())
df = pd.read_csv(CLUSTERED, low_memory=False)

df["helpful"] = df["reviews.numHelpful"].fillna(0)
df["title"] = df["reviews.title"].fillna("").astype(str)
df["text"] = df["reviews.text"].astype(str)

print("loaded %s -> %d reviews, %d categories"
      % (CLUSTERED, len(df), df["category"].nunique()))
print()
print(df.groupby("category")
      .agg(reviews=("text", "size"), rating=("reviews.rating", "mean"),
           negatives=("sentiment", lambda s: (s == "negative").sum()))
      .round(2).to_string())

loaded ..\datasets\reviews_clustered.csv -> 27866 reviews, 6 categories

                   reviews  rating  negatives
category                                     
Cases & covers          23    4.65          0
Chargers & cables      254    4.48         21
E-readers             4108    4.73         61
Smart speakers        3632    4.65         57
Streaming             2545    4.65         40
Tablets              17304    4.50        486


## 1. The facts — computed, not inferred

Everything the brief asks for by name. No model involved, so none of it can be wrong.

The "worst product" needs a minimum review count, otherwise a product with two reviews and
one complaint wins every time.

In [7]:
### vs Rayhan's cells 87 and 92 — he asked the MODEL for these.
#
# His prompts said "Mention: 1. What customers like 2. Common complaints
# 3. Overall customer opinion", and the brief also wants the top 3 products and
# the worst product. A language model cannot know any of that - it can only
# guess from the handful of reviews in the prompt. That is where the invented
# detail comes from, like the Echo summary that is actually about Google Home.
#
# Everything here is a groupby. It cannot be wrong, and it takes milliseconds.
MIN_REVIEWS_FOR_WORST = 20


def category_facts(d):
    '''Everything the brief asks for by name, straight from the data.'''
    per_product = (d.groupby("product_name")
                   .agg(reviews=("text", "size"),
                        rating=("reviews.rating", "mean"),
                        neg_rate=("sentiment", lambda s: (s == "negative").mean()))
                   .sort_values("reviews", ascending=False))

    # "Worst" means the product people actually complain about most, not the
    # one with the lowest mean star. In a category where every product scores
    # 4.4-4.8, picking the minimum rating just returns the best-seller - which
    # is what the first version of this notebook did for Tablets.
    eligible = per_product[per_product["reviews"] >= MIN_REVIEWS_FOR_WORST]
    cat_neg_rate = (d["sentiment"] == "negative").mean()

    worst = stands_out = None
    if len(eligible):
        worst = eligible["neg_rate"].idxmax()
        # only call it "worst" if it is meaningfully above the category average
        stands_out = eligible.loc[worst, "neg_rate"] > max(cat_neg_rate * 1.5, 0.01)

    return {
        "reviews": len(d),
        "rating": d["reviews.rating"].mean(),
        "pct_negative": 100 * cat_neg_rate,
        "pct_positive": 100 * (d["sentiment"] == "positive").mean(),
        "top3": per_product.head(3),
        "worst": worst,
        "worst_stands_out": stands_out,
        "worst_rating": eligible.loc[worst, "rating"] if worst is not None else None,
        "worst_negrate": eligible.loc[worst, "neg_rate"] if worst is not None else None,
        "worst_reviews": int(eligible.loc[worst, "reviews"]) if worst is not None else None,
        "n_products": len(per_product),
    }


FACTS = {c: category_facts(d) for c, d in df.groupby("category")}

for c, f in sorted(FACTS.items(), key=lambda kv: -kv[1]["reviews"]):
    print("=" * 78)
    print("%s  -  %s reviews, %d products, %.2f stars, %.1f%% negative"
          % (c, format(f["reviews"], ","), f["n_products"], f["rating"], f["pct_negative"]))
    print("   top 3 by review count:")
    for n, r in f["top3"].iterrows():
        print("      %6d  %.2f*  %s" % (r["reviews"], r["rating"], str(n)[:56]))
    if f["worst"] is not None:
        print("   most complained about (>=%d reviews): %.1f%% negative, %.2f*  %s%s"
              % (MIN_REVIEWS_FOR_WORST, 100 * f["worst_negrate"], f["worst_rating"],
                 str(f["worst"])[:46],
                 "" if f["worst_stands_out"] else "   (not notably worse than the rest)"))
    else:
        print("   no product has %d+ reviews in this category" % MIN_REVIEWS_FOR_WORST)

Tablets  -  17,304 reviews, 13 products, 4.50 stars, 2.8% negative
   top 3 by review count:
       10962  4.45*  Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special O
        2814  4.59*  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - I
        1685  4.51*  Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green
   most complained about (>=20 reviews): 3.7% negative, 4.51*  Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16   (not notably worse than the rest)
E-readers  -  4,108 reviews, 9 products, 4.73 stars, 1.5% negative
   top 3 by review count:
        3176  4.76*  Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monoc
         580  4.74*  Kindle Voyage E-reader, 6 High-Resolution Display (300 p
         212  4.43*  All-New Kindle E-reader - Black, 6 Glare-Free Touchscree
   most complained about (>=20 reviews): 8.0% negative, 4.43*  All-New Kindle E-reader - Black, 6 Glare-Free 
Smart speakers  -  3,632 reviews, 3 products, 4.65 stars, 1.6% negative
   top 3 by review

## 2. The complaints — measured by log-odds

To find what people complain about, compare how often a word appears in this category's
**negative** reviews against its **positive** ones.

Raw frequency does not work: "tablet" and "kindle" are the commonest words in both. The
log-odds ratio asks a better question — *how much more likely is this word in a complaint
than in praise?* — with Dirichlet smoothing so a word appearing twice cannot top the list.

This is measured from the actual reviews, so unlike a model's guess it can be checked.

In [8]:
### vs Rayhan's notebook — there is no equivalent cell.
#
# His prompt asked the model to find "common complaints" by reading 12 reviews.
# flan-t5-small returned one copied review, and the second prompt returned the
# instruction itself.
#
# Complaints are measurable instead. Compare how often a word appears in a
# category's NEGATIVE reviews against its POSITIVE ones. Raw frequency does not
# work - "tablet" and "kindle" top both lists - so this uses the log-odds ratio
# with Dirichlet smoothing, which asks how much MORE likely a word is in a
# complaint than in praise.
#
# The result can be checked against the data. A model's guess cannot.
STOP = set('''a about after all also am an and any are as at be been being but by can
cant could did do does doing dont for from had has have having he her here hers him his
how i if in into is it its just me more most my no nor not now of off on once only or
other our out over own same she should so some such than that the their them then there
these they this those through to too under until up very was we were what when where
which while who whom why will with you your amazon product one get got would im ive
very really much thing things'''.split())

TOKEN = re.compile(r"[a-z][a-z']{2,}")
ALPHA = 0.5          # Dirichlet smoothing


def complaint_terms(d, top_n=6):
    '''Words far more likely in this category's negative reviews than its positive ones.

    Two guards, both learned the hard way. The first version of this notebook
    counted every occurrence and required only 3, which for Tablets returned
    "wowwee", "robots" and "mip" - words from two reviews about a toy robot,
    repeated several times each. Log-odds rewards a word that appears ONLY in
    negatives, so a handful of mentions in 486 complaints scored enormously.

      - count DOCUMENT frequency, not term frequency, so one chatty review
        cannot invent a theme
      - scale the threshold to the category: a word must appear in at least 2%
        of its negative reviews, minimum 5
    '''
    def bag(subset):
        c = Counter()
        for t in (subset["title"] + " " + subset["text"]):
            # set() -> each review counts a word at most once
            c.update(set(w for w in TOKEN.findall(t.lower()) if w not in STOP))
        return c

    neg_docs = d[d["sentiment"] == "negative"]
    neg = bag(neg_docs)
    pos = bag(d[d["sentiment"] == "positive"])
    if len(neg_docs) < 25:
        return []                      # too few complaints to measure anything

    min_df = max(5, int(0.02 * len(neg_docs)))

    vocab = set(neg) | set(pos)
    n_neg, n_pos, V = sum(neg.values()), sum(pos.values()), len(vocab)

    scored = []
    for w in vocab:
        if neg[w] < min_df:
            continue
        lo = (np.log((neg[w] + ALPHA) / (n_neg + ALPHA * V))
              - np.log((pos[w] + ALPHA) / (n_pos + ALPHA * V)))
        scored.append((w, lo, neg[w]))

    return sorted(scored, key=lambda x: -x[1])[:top_n]


COMPLAINTS = {c: complaint_terms(d) for c, d in df.groupby("category")}

for c, terms in sorted(COMPLAINTS.items()):
    print("=" * 70)
    print(c)
    if not terms:
        n = (df[(df["category"] == c) & (df["sentiment"] == "negative")]).shape[0]
        print("   only %d negative reviews - too few to measure complaints" % n)
        continue
    n_neg = (df[(df["category"] == c) & (df["sentiment"] == "negative")]).shape[0]
    print("   (%d negative reviews, a word must appear in at least %d of them)"
          % (n_neg, max(5, int(0.02 * n_neg))))
    for w, lo, n in terms:
        print("   %-18s log-odds %+5.2f   in %d of the %d complaints" % (w, lo, n, n_neg))

Cases & covers
   only 0 negative reviews - too few to measure complaints
Chargers & cables
   only 21 negative reviews - too few to measure complaints
E-readers
   (61 negative reviews, a word must appear in at least 5 of them)
   return             log-odds +2.23   in 7 of the 61 complaints
   tried              log-odds +2.01   in 8 of the 61 complaints
   disappointed       log-odds +1.68   in 5 of the 61 complaints
   charger            log-odds +1.11   in 7 of the 61 complaints
   charge             log-odds +0.75   in 13 of the 61 complaints
   work               log-odds +0.56   in 5 of the 61 complaints
Smart speakers
   (57 negative reviews, a word must appear in at least 5 of them)
   screen             log-odds +1.59   in 5 of the 57 complaints
   went               log-odds +1.36   in 6 of the 57 complaints
   limited            log-odds +1.32   in 5 of the 57 complaints
   disappointed       log-odds +1.28   in 5 of the 57 complaints
   never              log-odds +1.21  

## 3. The evidence — the complaint other customers endorsed

`reviews.numHelpful` is populated for 98.6% of rows. Sorting negative reviews by it gives
the complaint that most readers found useful, instead of a random one.

This is the difference between Rayhan's *"Great product."* and a real, specific grievance.
Where a category has no helpful-voted negatives, fall back to the lowest-rated one and say
so rather than inventing something.

In [9]:
### vs Rayhan's cells 83-85 — he sampled 12 reviews at RANDOM.
#
#   his : .sample(n=min(positive_n, ...), random_state=42)
#         with positive_n=6, neutral_n=3, negative_n=3
#   here: nlargest(k, "helpful")
#
# A category holds thousands of reviews; twelve random ones cannot describe it.
# reviews.numHelpful is populated for 98.6% of rows, so sorting by it gives the
# complaint other customers endorsed rather than an arbitrary one.
#
# Same model, very different input. Random sampling gave him "Great product."
# This gives "Stay away from the certified refurbished Amazon Fire TV. I
# received two Certified Refurbished defective fire TVs" - 292 helpful votes.
def evidence(d, k=3):
    '''The most-endorsed complaints, or the lowest-rated ones if nobody voted.'''
    neg = d[d["sentiment"] == "negative"].copy()
    if neg.empty:
        return [], "none"
    if (neg["helpful"] > 0).any():
        picked = neg.nlargest(k, "helpful")
        how = "most helpful"
    else:
        picked = neg.nsmallest(k, "reviews.rating")
        how = "lowest rated (no helpful votes)"
    return [(int(r["helpful"]), r["text"].strip()) for _, r in picked.iterrows()], how


EVIDENCE = {}
for c, d in df.groupby("category"):
    EVIDENCE[c] = evidence(d)
    quotes, how = EVIDENCE[c]
    print("=" * 78)
    print("%s   (%s)" % (c, how))
    if not quotes:
        print("   no negative reviews at all in this category")
    for h, q in quotes:
        print("   [%3d helpful] %s" % (h, q[:150]))

Cases & covers   (none)
   no negative reviews at all in this category
Chargers & cables   (lowest rated (no helpful votes))
   [  0 helpful] Dont have option for password ask you before buying apps once you authorized as allow its done.Not as many free good apps Like apple have it
   [  0 helpful] Most buyers will have a phone or other charger that will work so you probably don't need to buy this.
   [  0 helpful] cheap - need to constantly make sure plugged in - slightly too small - saved some bucks there - not amazon quality
E-readers   (most helpful)
   [ 23 helpful] The Voyage, when it worked, was great - the page turns make the reading experience seamless and the ink is crisp, making it a real upgrade from the Pa
   [ 21 helpful] I eventually had to return it because of the shades of yellow in the back-light when using it at night. It was okay at day time but at night during wh
   [ 17 helpful] Very disappointed that my Kindle Voyage, purchased in April, failed after just two use

## 4. The model — used only where a model helps

Now, and only now, a language model. It is given the evidence quotes and asked to compress
them into one sentence. That is what a summarisation model is actually built for.

It is **not** asked what the top products are, how many reviews there were, or which
product is worst — those are already known exactly.

`sshleifer/distilbart-cnn-12-6` is a distilled BART, far stronger than the 80M
`flan-t5-small`. Set `USE_MODEL = False` to skip the download; the articles still build,
they just quote the evidence directly instead of compressing it.

In [10]:
### vs Rayhan's cells 86-97 — different model, different job, different settings.
#
# WHICH MODEL. He used google/flan-t5-small (80M), then switched to
# Falconsai/text_summarization. The first is too small to follow an instruction
# - his cell 94 proves it, the model answered "Overall opinion Do not copy
# individual reviews", which is his own prompt handed back. The second is a NEWS
# summariser: built to compress one long article by picking its best sentences.
# Our reviews average 158 characters, so it just returned the most quotable one.
# distilbart-cnn-12-6 is ~300M and genuinely abstractive.
#
# WHICH JOB. He gave the model the reviews and asked for facts and themes. Here
# it is given only the evidence quotes and asked to compress them into one
# sentence - the single thing a summarisation model is actually built for. It is
# never asked what the top products are; those are already known exactly.
#
# WHICH SETTINGS. He set max_new_tokens=150 with no minimum, so generation
# stopped after one sentence, and nothing prevented repetition - which is why
# one of his summaries reads "Kindle is a good choice for a Kindle."
USE_MODEL = True
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

summarise = None
if USE_MODEL:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        torch.set_num_threads(2)

        # NOTE: transformers 5.x REMOVED the "summarization" pipeline task -
        # pipeline("summarization", ...) now raises KeyError. The direct
        # seq2seq API below still works and is what Rayhan used in his cell 96.
        print("loading %s ..." % MODEL_NAME)
        _tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        _mod = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).eval()
        # the checkpoint ships min_length/max_length, which clash with the
        # new-token limits below and print a warning on every call
        _mod.generation_config.min_length = 0
        _mod.generation_config.max_length = None

        def summarise(text):
            enc = _tok(text, return_tensors="pt", truncation=True, max_length=512)
            with torch.no_grad():
                out = _mod.generate(
                    **enc,
                    min_new_tokens=25,        # a FLOOR - Rayhan had only a ceiling,
                    max_new_tokens=80,        # which is why his output was one line
                    no_repeat_ngram_size=3,   # stops "a Kindle is good for a Kindle"
                    num_beams=4,
                )
            return _tok.decode(out[0], skip_special_tokens=True).strip()

        print("ready - about 25 s per category on this CPU")
    except Exception as e:
        print("model unavailable (%s) - falling back to quoting directly"
              % type(e).__name__)
        summarise = None
else:
    print("USE_MODEL is False - articles will quote the evidence directly")

loading sshleifer/distilbart-cnn-12-6 ...


Loading weights: 100%|██████████| 358/358 [00:00<00:00, 6257.70it/s]


ready - about 25 s per category on this CPU


## 5. Assembling the articles

Facts first, then the measured complaints, then the evidence. The model contributes one
sentence, clearly marked in the code so nobody has to guess which part was generated.

In [11]:
### vs Rayhan's cells 88, 94 and 97 — the model writes ONE sentence, not the article.
#
# His design: give the model everything, take whatever comes back as the summary.
# This design: assemble the article from computed facts, and call the model once,
# on the last line, for the prose that needs judgement. The call is marked below
# so nobody has to guess which part was generated.
#
# If the model is unavailable the article still builds - it quotes the most
# endorsed complaint directly instead. Nothing here depends on generation
# succeeding.
def build_article(cat):
    f, terms, (quotes, how) = FACTS[cat], COMPLAINTS[cat], EVIDENCE[cat]
    L = []

    L.append("%s - %s reviews across %d products, averaging %.2f stars."
             % (cat, format(f["reviews"], ","), f["n_products"], f["rating"]))
    L.append("%.1f%% of reviews are positive and %.1f%% negative."
             % (f["pct_positive"], f["pct_negative"]))

    names = [str(n) for n in f["top3"].index]
    counts = [int(r) for r in f["top3"]["reviews"]]
    L.append("Most reviewed: " + "; ".join(
        "%s (%s reviews)" % (n[:60], format(c, ",")) for n, c in zip(names, counts)) + ".")

    if f["worst"] is not None and f["worst_stands_out"]:
        L.append("Most complained about: %s - %.1f%% of its %s reviews are negative, "
                 "against %.1f%% for the category, at %.2f stars."
                 % (str(f["worst"])[:60], 100 * f["worst_negrate"],
                    format(f["worst_reviews"], ","), f["pct_negative"],
                    f["worst_rating"]))
    elif f["worst"] is not None:
        L.append("No single product stands out as worse than the rest; the highest "
                 "negative rate is %.1f%% against %.1f%% for the category."
                 % (100 * f["worst_negrate"], f["pct_negative"]))

    if terms:
        L.append("Complaints centre on: "
                 + ", ".join("%s (in %d reviews)" % (w, n) for w, _, n in terms) + ".")
    else:
        L.append("Too few negative reviews here to identify complaint themes.")

    if quotes:
        joined = " ".join(q for _, q in quotes)
        if summarise is not None and len(joined.split()) > 25:
            L.append("What unhappy customers say: " + summarise(joined))   # <- the model
        else:
            h, q = quotes[0]
            L.append("The most endorsed complaint (%d readers found it helpful): \"%s\""
                     % (h, q[:220]))

    return " ".join(L)


ARTICLES = {}
for cat in sorted(FACTS, key=lambda c: -FACTS[c]["reviews"]):
    ARTICLES[cat] = build_article(cat)
    print("=" * 86)
    print(cat.upper())
    print("=" * 86)
    print(ARTICLES[cat])
    print()

[transformers] Both `min_new_tokens` (=25) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TABLETS
Tablets - 17,304 reviews across 13 products, averaging 4.50 stars. 91.7% of reviews are positive and 2.8% negative. Most reviewed: Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offer (10,962 reviews); All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Inclu (2,814 reviews); Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid (1,685 reviews). No single product stands out as worse than the rest; the highest negative rate is 3.7% against 2.8% for the category. Complaints centre on: worst (in 11 reviews), returning (in 23 reviews), waste (in 17 reviews), horrible (in 9 reviews), returned (in 47 reviews), barely (in 9 reviews). What unhappy customers say: Reviews are good. But, I realized they left reviews before they receive it . It is impossible to type & see Korean language. Do not buy this tablet if you are Korean. It is bad for web surfing.



[transformers] Both `min_new_tokens` (=25) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


E-READERS
E-readers - 4,108 reviews across 9 products, averaging 4.73 stars. 96.2% of reviews are positive and 1.5% negative. Most reviewed: Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrom (3,176 reviews); Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi)  (580 reviews); All-New Kindle E-reader - Black, 6 Glare-Free Touchscreen Di (212 reviews). Most complained about: All-New Kindle E-reader - Black, 6 Glare-Free Touchscreen Di - 8.0% of its 212 reviews are negative, against 1.5% for the category, at 4.43 stars. Complaints centre on: return (in 7 reviews), tried (in 8 reviews), disappointed (in 5 reviews), charger (in 7 reviews), charge (in 13 reviews), work (in 5 reviews). What unhappy customers say: The Kindle Voyage, when it worked, was great - the page turns make the reading experience seamless and the ink is crisp, making it a real upgrade from the Paperwhite . It started to freeze up quickly after I got it . I guess Amazon needs a better QA department .



[transformers] Both `min_new_tokens` (=25) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SMART SPEAKERS
Smart speakers - 3,632 reviews across 3 products, averaging 4.65 stars. 94.6% of reviews are positive and 1.6% negative. Most reviewed: Echo (White) (3,310 reviews); Amazon - Amazon Tap Portable Bluetooth and Wi-Fi Speaker - B (318 reviews); Echo (Black) (4 reviews). No single product stands out as worse than the rest; the highest negative rate is 1.6% against 1.6% for the category. Complaints centre on: screen (in 5 reviews), went (in 6 reviews), limited (in 5 reviews), disappointed (in 5 reviews), never (in 6 reviews), back (in 6 reviews). What unhappy customers say: Stay away from certified refurbished Amazon Fire TV . Amazon needs to re-visit its refurbishment process and quality control of the refurbished products .



[transformers] Both `min_new_tokens` (=25) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STREAMING
Streaming - 2,545 reviews across 5 products, averaging 4.65 stars. 94.9% of reviews are positive and 1.6% negative. Most reviewed: Amazon Fire Tv (2,528 reviews); Amazon Echo and Fire TV Power Adapter (8 reviews); Certified Refurbished Amazon Fire TV with Alexa Voice Remote (4 reviews). No single product stands out as worse than the rest; the highest negative rate is 1.6% against 1.6% for the category. Complaints centre on: returned (in 8 reviews), disappointed (in 6 reviews), money (in 6 reviews), didn't (in 7 reviews), understand (in 5 reviews), connect (in 5 reviews). What unhappy customers say: Bought this as a present for my boyfriend, and we both were very disappointed . Took some getting use to for me to get the commands right to change my nest thermostat . Must have Amazon paid service to play music from the cloud and must be set up from the mobile app .



[transformers] Both `min_new_tokens` (=25) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CHARGERS & CABLES
Chargers & cables - 254 reviews across 4 products, averaging 4.48 stars. 87.4% of reviews are positive and 8.3% negative. Most reviewed: Amazon 5W USB Official OEM Charger and Power Adapter for Fir (207 reviews); Amazon 9W PowerFast Official OEM USB Charger and Power Adapt (36 reviews); New Amazon Kindle Fire Hd 9w Powerfast Adapter Charger + Mic (7 reviews). No single product stands out as worse than the rest; the highest negative rate is 10.1% against 8.3% for the category. Too few negative reviews here to identify complaint themes. What unhappy customers say: Most buyers will have a phone or other charger that will work so you probably don't need to buy this . Need to constantly make sure plugged in - slightly too small . Not as many free good apps like apple have it .

CASES & COVERS
Cases & covers - 23 reviews across 5 products, averaging 4.65 stars. 95.7% of reviews are positive and 0.0% negative. Most reviewed: Kindle Dx Leather Cover, Black (fits 9.7 Display, 

## 6. Saving in the format the app already reads

`app.py` loads `category_summaries.json` as `{category: text}` and shows it with
`st.info(summary)`. Writing the same shape means **the app does not change at all** —
replace the file and the dashboard improves.

In [12]:
### vs Rayhan's app.py — deliberately the SAME format, so his app is untouched.
#
# app.py does:   summaries = json.load(open("category_summaries.json"))
#                st.info(summaries.get(category, "No summary available."))
#
# Writing {category: text} means the dashboard picks these up with zero code
# change. The blue "AI Customer Summary" box just gets better content.
OUT_JSON = Path("category_summaries.json")
with open(OUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(ARTICLES, fh, indent=2, ensure_ascii=False)

webapp = Path("../webapp/category_summaries.json")
if webapp.parent.exists():
    with open(webapp, "w", encoding="utf-8") as fh:
        json.dump(ARTICLES, fh, indent=2, ensure_ascii=False)
    print("also written to", webapp)

print("saved %s - %d categories" % (OUT_JSON, len(ARTICLES)))
print()
print("average length: %d characters (Rayhan's averaged 74)"
      % int(np.mean([len(v) for v in ARTICLES.values()])))

also written to ..\webapp\category_summaries.json
saved category_summaries.json - 6 categories

average length: 772 characters (Rayhan's averaged 74)


## 7. Side by side with the earlier attempt

The same category, before and after.

In [13]:
OLD = {
    "Fire HD Tablets": "I bought this for my grandson and he loves it. great",
    "Accessories & Power": "Kindle is a good choice for a Kindle.",
    "Echo & Smart Home": ("I have two Google home devices. They seem to be lacking in "
                          "smart5 home controls. ... I expect that Google home will surpas"),
}

print("BEFORE - flan-t5-small / Falconsai, prompt only")
print("=" * 86)
for k, v in OLD.items():
    print("%-22s %s" % (k, v))
print()
print("AFTER - facts computed, complaints measured, quotes endorsed")
print("=" * 86)
for k in list(ARTICLES)[:3]:
    print("%s\n   %s\n" % (k, ARTICLES[k]))

BEFORE - flan-t5-small / Falconsai, prompt only
Fire HD Tablets        I bought this for my grandson and he loves it. great
Accessories & Power    Kindle is a good choice for a Kindle.
Echo & Smart Home      I have two Google home devices. They seem to be lacking in smart5 home controls. ... I expect that Google home will surpas

AFTER - facts computed, complaints measured, quotes endorsed
Tablets
   Tablets - 17,304 reviews across 13 products, averaging 4.50 stars. 91.7% of reviews are positive and 2.8% negative. Most reviewed: Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offer (10,962 reviews); All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Inclu (2,814 reviews); Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid (1,685 reviews). No single product stands out as worse than the rest; the highest negative rate is 3.7% against 2.8% for the category. Complaints centre on: worst (in 11 reviews), returning (in 23 reviews), waste (in 17 reviews), horrible (in 9 re

## 8. What this notebook establishes

- **The brief's three requirements are now actually answered.** Top 3 products, common
  complaints and the worst product all appear in every article, and all three are computed
  rather than inferred.
- **Nothing can be hallucinated.** Every number comes from `groupby`, every complaint word
  from a log-odds count, every quote from a real review with its helpful-vote count. The
  model only ever sees text that is already true.
- **The model is used for what it is good at** — compressing several sentences into one —
  and never asked for facts.

### Honest limits

- **Two categories are too small to summarise properly.** `Cases & covers` has 23 reviews
  and **zero** negatives; `Chargers & cables` has 21 negatives but none with helpful votes.
  The code says so rather than inventing complaints. Merging the two accessory categories
  would fix it, at the cost of a less specific category.
- **Log-odds finds words, not themes.** "charger" and "battery" are real signals;
  turning them into a sentence is still the model's job, and it can still phrase that
  badly.
- **`reviews.numHelpful` reflects what readers upvoted**, which is not the same as what is
  most common — a dramatic complaint attracts votes. It is better evidence than a random
  sample, not perfect evidence.